# Контрольна робота: вартість життя у світі

**Тема 6**  
**Група:** ПВ512

## 1. Підготовка середовища

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", context="notebook")

DATA_FILE = Path("CostofLiving.csv")
OUTPUT_DIR = Path("results")
OUTPUT_DIR.mkdir(exist_ok=True)

## 2. Завантаження та первинний огляд даних

In [2]:
df = pd.read_csv(DATA_FILE)

print(f"Розмір датасету: {df.shape[0]} рядків, {df.shape[1]} стовпців")
print("\nТипи даних:")
print(df.dtypes)
print("\nПерші 5 рядків:")
print(df.head())
print("\nОстанні 5 рядків:")
print(df.tail())
print("\nОпис числових показників:")
print(df.describe().round(2))
print("\nПропущені значення:")
print(df.isna().sum())
print(f"\nПовних дублікатів: {df.duplicated().sum()}")

Розмір датасету: 540 рядків, 7 стовпців

Типи даних:
City                                  str
Cost of Living Index              float64
Rent Index                        float64
Cost of Living Plus Rent Index    float64
Groceries Index                   float64
Restaurant Price Index            float64
Local Purchasing Power Index      float64
dtype: object

Перші 5 рядків:
                  City  Cost of Living Index  Rent Index  Cost of Living Plus Rent Index  Groceries Index  Restaurant Price Index  \
0    Hamilton, Bermuda                145.43      110.87                          128.76           143.47                  158.75   
1  Zurich, Switzerland                141.25       66.14                          105.03           149.86                  135.76   
2  Geneva, Switzerland                134.83       71.70                          104.38           138.98                  129.74   
3   Basel, Switzerland                130.68       49.68                           91.61  

Датасет містить міста та індекси вартості життя, оренди, продуктів, ресторанів і місцевої купівельної спроможності. Первинний огляд дає змогу перевірити структуру, якість та діапазони значень перед аналізом.

## 3. Очищення та підготовка

In [3]:
numeric_columns = df.columns.drop("City")
df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric, errors="coerce")

missing_before = int(df.isna().sum().sum())
duplicates_before = int(df.duplicated(subset="City").sum())

df = df.drop_duplicates(subset="City", keep="first").copy()
for column in numeric_columns:
    df[column] = df[column].fillna(df[column].median())

df["Country"] = df["City"].str.rsplit(",", n=1).str[-1].str.strip()
df["level"] = pd.cut(
    df["Cost of Living Index"],
    bins=[-np.inf, 40, 80, np.inf],
    labels=["Cheap", "Average", "Expensive"],
    right=False
)

index_columns = [
    "Cost of Living Index",
    "Rent Index",
    "Groceries Index",
    "Restaurant Price Index",
    "Local Purchasing Power Index"
]
df["total_index"] = df[index_columns].mean(axis=1)
df["purchasing_efficiency"] = df["Local Purchasing Power Index"] / df["Cost of Living Index"]

print(f"Пропусків до очищення: {missing_before}")
print(f"Дублікатів міст до очищення: {duplicates_before}")
print(f"Пропусків після очищення: {int(df.isna().sum().sum())}")
print(f"Розмір очищеного датасету: {df.shape}")
print("\nРозподіл міст за рівнем вартості життя:")
print(df["level"].value_counts().sort_index())

Пропусків до очищення: 0
Дублікатів міст до очищення: 0
Пропусків після очищення: 0
Розмір очищеного датасету: (540, 11)

Розподіл міст за рівнем вартості життя:
level
Cheap         98
Average      327
Expensive    115
Name: count, dtype: int64


Числові поля приведено до числового типу, пропуски заповнено медіанами, а повтори міст вилучено. Додано країну, категорію рівня вартості життя, середній індекс і показник купівельної ефективності.

## 4. Обов'язковий аналіз

### 4.1. Топ-10 найдорожчих міст

In [4]:
top_expensive = df.nlargest(10, "Cost of Living Index")[["City", "Cost of Living Index"]]
print(top_expensive.to_string(index=False))
print(f"\nВисновок: найвищий індекс має {top_expensive.iloc[0]['City']} — {top_expensive.iloc[0]['Cost of Living Index']:.2f}.")

                 City  Cost of Living Index
    Hamilton, Bermuda                145.43
  Zurich, Switzerland                141.25
  Geneva, Switzerland                134.83
   Basel, Switzerland                130.68
    Bern, Switzerland                128.03
Lausanne, Switzerland                127.50
   Reykjavik, Iceland                123.78
    Stavanger, Norway                118.61
  Lugano, Switzerland                118.24
         Oslo, Norway                117.23

Висновок: найвищий індекс має Hamilton, Bermuda — 145.43.


### 4.2. Топ-10 найдешевших міст

In [5]:
top_cheap = df.nsmallest(10, "Cost of Living Index")[["City", "Cost of Living Index"]]
print(top_cheap.to_string(index=False))
print(f"\nВисновок: найнижчий індекс має {top_cheap.iloc[0]['City']} — {top_cheap.iloc[0]['Cost of Living Index']:.2f}.")

                     City  Cost of Living Index
Thiruvananthapuram, India                 20.86
       Navi Mumbai, India                 23.44
        Alexandria, Egypt                 23.78
        Coimbatore, India                 24.61
             Kochi, India                 24.65
     Visakhapatnam, India                 24.66
         Kharkiv, Ukraine                 24.85
       Bhubaneswar, India                 24.89
            Mysore, India                 25.20
            Lviv, Ukraine                 25.31

Висновок: найнижчий індекс має Thiruvananthapuram, India — 20.86.


### 4.3. Зв'язок вартості життя та оренди

In [6]:
cost_rent_corr = df["Cost of Living Index"].corr(df["Rent Index"])
strength = "сильний" if abs(cost_rent_corr) >= 0.7 else "помірний" if abs(cost_rent_corr) >= 0.4 else "слабкий"
direction = "прямий" if cost_rent_corr > 0 else "обернений"
print(f"Кореляція Пірсона: {cost_rent_corr:.3f}")
print(f"Висновок: між вартістю життя й орендою спостерігається {strength} {direction} зв'язок.")

Кореляція Пірсона: 0.788
Висновок: між вартістю життя й орендою спостерігається сильний прямий зв'язок.


### 4.4. Міста з найвищою купівельною спроможністю

In [7]:
top_power = df.nlargest(10, "Local Purchasing Power Index")[["City", "Local Purchasing Power Index"]]
print(top_power.to_string(index=False))
print(f"\nВисновок: лідером є {top_power.iloc[0]['City']} з індексом {top_power.iloc[0]['Local Purchasing Power Index']:.2f}.")

                          City  Local Purchasing Power Index
Sioux Falls, SD, United States                        168.93
  Ann Arbor, MI, United States                        163.57
     Dallas, TX, United States                        162.37
              Hanover, Germany                        160.86
  Rochester, NY, United States                        158.71
             Mannheim, Germany                        156.77
San Antonio, TX, United States                        154.92
    Memphis, TN, United States                        154.91
    Atlanta, GA, United States                        153.77
  Arlington, TX, United States                        152.90

Висновок: лідером є Sioux Falls, SD, United States з індексом 168.93.


### 4.5. Категорії вартості життя

In [8]:
level_summary = df.groupby("level", observed=False).agg(
    cities=("City", "count"),
    average_cost=("Cost of Living Index", "mean"),
    average_rent=("Rent Index", "mean")
).round(2)
print(level_summary)
print(f"\nВисновок: найбільше міст належить до категорії {level_summary['cities'].idxmax()}.")

           cities  average_cost  average_rent
level                                        
Cheap          98         32.88          8.95
Average       327         63.43         26.17
Expensive     115         90.48         45.56

Висновок: найбільше міст належить до категорії Average.


### 4.6. Загальний середній індекс

In [9]:
top_total = df.nlargest(10, "total_index")[["City", "total_index"]]
print(top_total.to_string(index=False))
print(f"\nВисновок: за сукупністю п'яти показників перше місце посідає {top_total.iloc[0]['City']}.")

                 City  total_index
    Hamilton, Bermuda      134.156
  Zurich, Switzerland      127.142
  Geneva, Switzerland      121.242
   Basel, Switzerland      114.826
Lausanne, Switzerland      113.296
    Bern, Switzerland      107.298
  Lugano, Switzerland      106.210
    Stavanger, Norway      105.242
   Reykjavik, Iceland      104.264
         Oslo, Norway      101.192

Висновок: за сукупністю п'яти показників перше місце посідає Hamilton, Bermuda.


### 4.7. Дешева оренда та дорогі продукти

In [10]:
rent_limit = df["Rent Index"].quantile(0.25)
groceries_limit = df["Groceries Index"].quantile(0.75)
cheap_rent_expensive_food = df[
    (df["Rent Index"] <= rent_limit) &
    (df["Groceries Index"] >= groceries_limit)
][["City", "Rent Index", "Groceries Index"]].sort_values("Groceries Index", ascending=False)

print(f"Межа дешевої оренди: {rent_limit:.2f}")
print(f"Межа дорогих продуктів: {groceries_limit:.2f}")
print(cheap_rent_expensive_food.to_string(index=False) if not cheap_rent_expensive_food.empty else "Міст за заданими умовами не знайдено.")
print(f"\nВисновок: критеріям одночасно відповідає {len(cheap_rent_expensive_food)} міст.")

Межа дешевої оренди: 13.01
Межа дорогих продуктів: 74.41
            City  Rent Index  Groceries Index
Taichung, Taiwan        13.0            84.29

Висновок: критеріям одночасно відповідає 1 міст.


### 4.8. Ресторанні ціни та купівельна спроможність

In [11]:
restaurant_power_corr = df["Restaurant Price Index"].corr(df["Local Purchasing Power Index"])
restaurant_power = df.groupby("level", observed=False)[
    ["Restaurant Price Index", "Local Purchasing Power Index"]
].mean().round(2)
print(restaurant_power)
print(f"\nКореляція: {restaurant_power_corr:.3f}")
print("Висновок: додатне значення означає, що в містах із дорожчими ресторанами купівельна спроможність у середньому також вища; від'ємне означало б протилежну тенденцію.")

           Restaurant Price Index  Local Purchasing Power Index
level                                                          
Cheap                       25.22                         52.53
Average                     58.33                         95.14
Expensive                   89.88                        108.06

Кореляція: 0.652
Висновок: додатне значення означає, що в містах із дорожчими ресторанами купівельна спроможність у середньому також вища; від'ємне означало б протилежну тенденцію.


## 5. Додатковий аналіз

In [12]:
country_summary = df.groupby("Country").agg(
    cities=("City", "count"),
    average_cost=("Cost of Living Index", "mean"),
    average_power=("Local Purchasing Power Index", "mean"),
    average_efficiency=("purchasing_efficiency", "mean")
)
country_summary = country_summary[country_summary["cities"] >= 3]
best_efficiency = country_summary.nlargest(10, "average_efficiency").round(2)
print("Країни з найкращим співвідношенням купівельної спроможності до вартості життя:")
print(best_efficiency.to_string())
print(f"\nВисновок: серед країн, представлених щонайменше трьома містами, найкраще співвідношення має {best_efficiency.index[0]}.")

Країни з найкращим співвідношенням купівельної спроможності до вартості життя:
                      cities  average_cost  average_power  average_efficiency
Country                                                                      
India                     28         27.65          75.22                2.73
Saudi Arabia               4         49.56         127.01                2.57
South Africa               5         48.83          97.78                2.00
United Arab Emirates       3         65.64         126.09                1.93
China                      6         45.71          79.18                1.77
Mexico                    10         34.58          61.24                1.76
Germany                   18         75.40         128.19                1.71
United States            132         74.85         124.27                1.69
Czech Republic             4         48.72          77.21                1.59
Canada                    30         73.72         114.40      

## 6. Візуалізація

### 6.1. Топ-15 найдорожчих міст

In [13]:
top15 = df.nlargest(15, "Cost of Living Index").sort_values("Cost of Living Index")
fig, ax = plt.subplots(figsize=(11, 7))
sns.barplot(data=top15, x="Cost of Living Index", y="City", hue="City", palette="rocket", legend=False, ax=ax)
ax.set_title("Топ-15 найдорожчих міст", fontsize=15, fontweight="bold")
ax.set_xlabel("Індекс вартості життя")
ax.set_ylabel("")
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f", padding=3, fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "01_top15_expensive.png", dpi=180, bbox_inches="tight")
plt.show()

C:\Users\Admin\AppData\Local\Temp\ipykernel_8072\3783180210.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 6.2. Вартість життя та купівельна спроможність

In [14]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(
    data=df,
    x="Cost of Living Index",
    y="Local Purchasing Power Index",
    hue="level",
    size="Rent Index",
    sizes=(20, 180),
    alpha=0.75,
    palette="Set2",
    ax=ax
)
ax.set_title("Вартість життя та місцева купівельна спроможність", fontsize=14, fontweight="bold")
ax.set_xlabel("Індекс вартості життя")
ax.set_ylabel("Індекс купівельної спроможності")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "02_cost_vs_power.png", dpi=180, bbox_inches="tight")
plt.show()

C:\Users\Admin\AppData\Local\Temp\ipykernel_8072\2908894745.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 6.3. Розподіл індексу оренди

In [15]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(data=df, x="Rent Index", bins=25, kde=True, color="#4C72B0", ax=ax)
ax.axvline(df["Rent Index"].median(), color="#C44E52", linestyle="--", label=f"Медіана: {df['Rent Index'].median():.1f}")
ax.set_title("Розподіл індексу оренди", fontsize=14, fontweight="bold")
ax.set_xlabel("Індекс оренди")
ax.set_ylabel("Кількість міст")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "03_rent_distribution.png", dpi=180, bbox_inches="tight")
plt.show()

C:\Users\Admin\AppData\Local\Temp\ipykernel_8072\2182948177.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 6.4. Кореляція індексів

In [16]:
correlation = df[index_columns].corr()
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(correlation, annot=True, fmt=".2f", cmap="RdYlBu_r", vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax)
ax.set_title("Кореляція основних індексів", fontsize=14, fontweight="bold", pad=15)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "04_index_correlation.png", dpi=180, bbox_inches="tight")
plt.show()

C:\Users\Admin\AppData\Local\Temp\ipykernel_8072\998143954.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 6.5. Спільне полотно 2×2

In [17]:
fig, axes = plt.subplots(2, 2, figsize=(17, 13))

sns.barplot(data=top15, x="Cost of Living Index", y="City", hue="City", palette="rocket", legend=False, ax=axes[0, 0])
axes[0, 0].set_title("Топ-15 найдорожчих міст", fontweight="bold")
axes[0, 0].set_xlabel("Індекс вартості життя")
axes[0, 0].set_ylabel("")

sns.scatterplot(data=df, x="Cost of Living Index", y="Local Purchasing Power Index", hue="level", alpha=0.7, palette="Set2", ax=axes[0, 1])
axes[0, 1].set_title("Вартість життя та купівельна спроможність", fontweight="bold")
axes[0, 1].set_xlabel("Індекс вартості життя")
axes[0, 1].set_ylabel("Купівельна спроможність")

sns.histplot(data=df, x="Rent Index", bins=25, kde=True, color="#4C72B0", ax=axes[1, 0])
axes[1, 0].axvline(df["Rent Index"].median(), color="#C44E52", linestyle="--")
axes[1, 0].set_title("Розподіл індексу оренди", fontweight="bold")
axes[1, 0].set_xlabel("Індекс оренди")
axes[1, 0].set_ylabel("Кількість міст")

sns.heatmap(correlation, annot=True, fmt=".2f", cmap="RdYlBu_r", vmin=-1, vmax=1, square=True, cbar=False, ax=axes[1, 1])
axes[1, 1].set_title("Кореляція основних індексів", fontweight="bold")
axes[1, 1].tick_params(axis="x", rotation=35)

fig.suptitle("Вартість життя у містах світу", fontsize=18, fontweight="bold", y=1.01)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "05_dashboard_2x2.png", dpi=180, bbox_inches="tight")
plt.show()

C:\Users\Admin\AppData\Local\Temp\ipykernel_8072\825964300.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Збереження результатів

In [18]:
df.to_csv(OUTPUT_DIR / "cost_of_living_cleaned.csv", index=False, encoding="utf-8-sig")
top_expensive.to_csv(OUTPUT_DIR / "top_10_expensive.csv", index=False, encoding="utf-8-sig")
top_cheap.to_csv(OUTPUT_DIR / "top_10_cheapest.csv", index=False, encoding="utf-8-sig")
best_efficiency.to_csv(OUTPUT_DIR / "country_efficiency.csv", encoding="utf-8-sig")

saved_files = sorted(path.name for path in OUTPUT_DIR.iterdir())
print("Збережені файли:")
print("\n".join(saved_files))

Збережені файли:
01_top15_expensive.png
02_cost_vs_power.png
03_rent_distribution.png
04_index_correlation.png
05_dashboard_2x2.png
cost_of_living_cleaned.csv
country_efficiency.csv
top_10_cheapest.csv
top_10_expensive.csv


## Підсумок

Проведено повну розвідку й очищення даних, виконано всі вісім завдань теми, створено три обчислювані показники та п'ять графічних файлів. Додатково проаналізовано співвідношення купівельної спроможності й вартості життя на рівні країн.